# Model B — Skor Keausan Kontekstual dan Jadwal Servis

Data sensor lokal belum membawa nilai load-cell maupun label servis. Karena itu bagian pertama memakai skor yang transparan untuk skenario muatan yang ditulis eksplisit. Bagian kedua melatih benchmark RUL EVIoT yang dipisahkan dari inferensi perjalanan lokal.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from aic import WearConfig, score_wear, write_wear_model_spec, write_wear_outputs
from aic.reporting import write_model_b_presentation
from aic.synthetic import SyntheticLoadConfig, make_synthetic_load_profile

segments = pd.read_csv(PROJECT_ROOT / 'reports' / 'model_a' / 'segments.csv').query("vehicle == 'tt'").copy()
# Fixture ini hanya untuk debugging; ganti dengan pembacaan load-cell aktual.
scenario = WearConfig(load_kg=4000, load_limit_kg=8000, service_interval_km=10_000)
load_profile = make_synthetic_load_profile(segments, SyntheticLoadConfig(load_limit_kg=scenario.load_limit_kg))
(PROJECT_ROOT / 'data' / 'synthetic').mkdir(parents=True, exist_ok=True)
load_profile.to_csv(PROJECT_ROOT / 'data' / 'synthetic' / 'load_profile_demo.csv', index=False)
wear_report = score_wear(segments, scenario, load_profile)
write_wear_outputs(wear_report, PROJECT_ROOT / 'reports' / 'model_b', scenario)
write_wear_model_spec(PROJECT_ROOT / 'models' / 'model_b', scenario)
write_model_b_presentation(wear_report, segments, scenario, PROJECT_ROOT / 'reports' / 'model_b')
wear_report.session_summary

In [ ]:
from aic.benchmark import load_eviot_dataset, train_eviot_rul_benchmark, write_benchmark_outputs

eviot = load_eviot_dataset(PROJECT_ROOT / 'data' / 'external' / 'eviot_predictive_maintenance.zip')
benchmark = train_eviot_rul_benchmark(eviot)
write_benchmark_outputs(benchmark, PROJECT_ROOT / 'reports' / 'model_b_benchmark')
benchmark.metrics